# 02-3. 추가 자료: ERA5-Land 재분석과 서해 해기차

02의 6장입니다. ECMWF ERA5-Land 재분석의 격자 시계열과 서해 해기차를 분석 시간축에 맞춥니다. 공개된 공공 기상자료는 출처와 활용 근거를 밝히면 학습에 쓸 수 있습니다(QnA 2026-10-07).

ERA5-Land는 수치 모델과 관측 동화로 만든 재분석이고 해상도는 약 9km(0.1°)입니다. 도로기상(RWIS) 관측은 동화에 쓰이지 않아서 검증 지점의 관측이 입력에 섞이지 않습니다.

| 항목 | 내용 |
|------|------|
| 자료 | ERA5-Land hourly time series (Copernicus Climate Change Service, Climate Data Store `reanalysis-era5-land-timeseries`) |
| 이용 조건 | CC-BY 4.0, 출처 표기 필요 (Muñoz-Sabater et al. 2021) |
| 범위 | 구간 선이 지나는 0.1° 격자 27개 (구간 중간점이 쓰는 25개, 관측소가 쓰는 13개 포함), 2024-10-25 09시 ~ 2026-03-01 08시(KST), 1시간 |
| 변수 | 2m 기온·이슬점, 지표면온도(skin), 지표 기압, 시간당 강수·강설(물 환산), 하향 단파·장파 복사, 10m 바람 |
| 위치 | `data/external/era5land_cells/era5land_cell_<위도>_<경도>.csv` (격자별 14열). 저장소에는 없고 받는 방법은 README 3장에 있음 |

쓰는 곳마다 교차검증으로 기여를 쟀습니다. 03-2의 3-6-8(노면온도 밑그림), 04-2의 2-3(결빙 분류), 05-1의 2장(결빙 영향인자), 05-2의 3장(구간 강설 지표 M3)입니다. 서해 해기차는 6-4에서 설명합니다.

`data/processed/`에 네 파일을 만듭니다. 격자 연결과 시간축 정렬 함수는 `src/external.py`에 있습니다.

| 파일 | 내용 |
|------|------|
| `era5_10min.parquet` | 관측소 17곳, 10분 |
| `era5_segment_cells.csv` | 구간-격자 대응과 확보 여부 |
| `segment_era5_1h.parquet` | 구간, 1시간 |
| `sea_10min.parquet` | 서해 해기차, 10분 |

참고 자료: Muñoz-Sabater, J., et al. (2021). ERA5-Land: a state-of-the-art global reanalysis dataset for land applications. *Earth System Science Data*, 13, 4349–4383.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]  # notebooks/<단계 폴더>/ 에서 실행
sys.path.insert(0, str(ROOT))

from src import config

In [2]:
import warnings

import geopandas as gpd
import pandas as pd

from src import external as ex

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

stations = pd.read_parquet(config.PROCESSED_DIR / "stations.parquet").set_index("stn")


## 6. ERA5-Land 정렬

### 6-1. 파일과 격자

파일에 들어 있는 격자 중심(`grid_lat`, `grid_lon`)이 파일 이름의 격자와 같은지 확인했습니다. 관측소 좌표에서 가장 가까운 0.1° 격자를 고르고 구간(6-3)에도 같은 규칙을 씁니다.

In [3]:
raw, rows = {}, []
for cell in ex.available_cells():
    e = ex.load_cell(cell)
    assert (round(float(e["grid_lat"].iloc[0]), 1), round(float(e["grid_lon"].iloc[0]), 1)) == cell, cell  # 파일 이름과 같아야 한다
    rows.append({"격자": cell, "행": len(e), "시작": e.index.min(), "끝": e.index.max(), "결측": int(e.isna().sum().sum())})
    raw[cell] = e.drop(columns=["grid_lat", "grid_lon"])
files = pd.DataFrame(rows)
st_cell = pd.Series({s: ex.cell_of(r.lat, r.lon) for s, r in stations.iterrows()})
print(f"격자 파일 {len(files)}개, 관측소가 쓰는 격자 {st_cell.nunique()}개, 기간 {files['시작'].min()} ~ {files['끝'].max()}, "
      f"결측 {files['결측'].sum()}")
shared = pd.DataFrame({"지점명": stations["name"], "격자": st_cell, "검증 지점": stations["holdout"]})
shared["같은 격자 지점 수"] = shared.groupby("격자")["지점명"].transform("size")
shared[shared["같은 격자 지점 수"] > 1].sort_values("격자")


격자 파일 27개, 관측소가 쓰는 격자 13개, 기간 2024-10-25 09:00:00 ~ 2026-03-01 08:00:00, 결측 0


,지점명,격자,검증 지점,같은 격자 지점 수
33411,삼봉(도),"(35.7, 127.6)",True,2
33412,대곡(도),"(35.7, 127.6)",False,2
23203,삼례졸음(도),"(35.9, 127.1)",False,2
23402,해전(도),"(35.9, 127.1)",True,2
12103,군산휴(도),"(36.0, 126.8)",False,2
12409,우곡3교(도),"(36.0, 126.8)",False,2
33206,용포1교(도),"(36.0, 127.6)",False,2
33409,가당2교(도),"(36.0, 127.6)",False,2


관측소 17곳이 격자 13개에 들어갑니다. 검증 지점인 해전·삼봉은 학습 지점인 삼례졸음·대곡과 ERA5 입력이 같습니다. 공유하는 것은 입력이고 관측(라벨)은 아니어서 누수는 없습니다.

### 6-2. 10분 시간축 정렬

순간값(기온, 이슬점, 지표면온도, 풍속)은 시각 사이를 선형 보간합니다. 복사, 강수, 강설은 ERA5 규약상 그 시각에 끝나는 1시간 합계입니다(09시 값은 08~09시). 값을 시간 가운데(08:30)에 두고 선형 보간해서 10분마다 시간당 비율(mm/h, W/m²)로 씁니다.

02-1의 ASOS 보간과 같은 규칙입니다. 팀 명세서는 직전 값 유지(ffill)를 제안했습니다. 그 방식은 10:00~10:50에 09~10시 합계를 그대로 써서 합계 구간의 가운데(09:30)보다 평균 약 55분 늦어지고 02-1과 규칙도 달라서 바꿨습니다.

강수와 강설은 지난 3시간 합(mm)과 강설 비율(강설 / 강수, 지난 3시간 강수 0.1mm 이상일 때)도 만듭니다. UTC에서 KST로의 변환은 하향 단파 복사의 평균이 정오 무렵에 가장 큰지로 확인했습니다.

In [4]:
TIMES, PERIOD_ID = ex.analysis_times()
frames = [ex.add_history(ex.align(raw[st_cell[s]], TIMES), PERIOD_ID).assign(stn=s).rename_axis("time").reset_index()
          for s in stations.index]
ERA5_COLS = [*ex.INST.values(), *ex.ACC.values(), "tp3h", "sf3h", "snowfrac3h"]
era5 = pd.concat(frames, ignore_index=True)[["stn", "time", *ERA5_COLS]]
era5[ERA5_COLS] = era5[ERA5_COLS].astype("float32")
peak = raw[st_cell.iloc[0]]["ssrd_wm2"].groupby(lambda t: t.hour).mean().idxmax()
print(f"하향 단파 복사 평균이 가장 큰 시각: {peak}시 (KST, 그 시각에 끝나는 1시간)")
print(era5.shape, "결측", int(era5[ERA5_COLS].isna().sum().sum()))
era5[ERA5_COLS].describe().T.round(2)


하향 단파 복사 평균이 가장 큰 시각: 13시 (KST, 그 시각에 끝나는 1시간)
(587520, 13) 결측 0


,count,mean,std,min,25%,50%,75%,max
t2m,587520.0,2.44,5.90,-16.68,-1.80,1.76,6.28,21.96
d2m,587520.0,-3.06,6.89,-27.73,-7.91,-3.09,1.42,15.51
skt,587520.0,2.02,6.77,-22.37,-2.94,1.56,6.58,24.09
ws10,587520.0,2.75,1.66,0.02,1.50,2.36,3.65,11.12
tp_mmh,587520.0,0.04,0.19,0.00,0.00,0.00,0.01,5.40
sf_mmh,587520.0,0.01,0.07,0.00,0.00,0.00,0.00,2.50
ssrd,587520.0,123.18,186.15,0.00,0.00,0.03,226.37,773.98
strd,587520.0,257.76,38.28,156.73,228.78,254.88,283.20,395.25
tp3h,587520.0,0.12,0.56,0.00,0.00,0.00,0.03,14.71
sf3h,587520.0,0.04,0.21,0.00,0.00,0.00,0.00,6.74


02-2의 AWS 보간값과 비교해 자료가 제대로 붙었는지 봤습니다. 입력끼리의 비교라 관측 라벨은 쓰지 않습니다. 강수 일치율은 AWS가 강수를 감지한 10분 가운데 ERA5 강수(0.1 mm/h 이상)도 있는 비율입니다.

In [5]:
sfe = pd.read_parquet(config.PROCESSED_DIR / "station_features.parquet", columns=["stn", "time", "ta", "rain"])
m = sfe.merge(era5[["stn", "time", "t2m", "tp_mmh"]], on=["stn", "time"])
aws_rain, e5_rain = m["rain"] >= 0.5, m["tp_mmh"] > 0.1
check = pd.DataFrame({
    "기온 상관": m.groupby("stn").apply(lambda g: g["ta"].corr(g["t2m"])),
    "기온 차 (ERA5 - AWS 보간, °C)": (m["t2m"] - m["ta"]).groupby(m["stn"]).mean(),
    "강수 일치율": (aws_rain & e5_rain).groupby(m["stn"]).sum() / aws_rain.groupby(m["stn"]).sum(),
    "ERA5만 강수인 시간 비율": (e5_rain & ~aws_rain).groupby(m["stn"]).mean(),
})
check.insert(0, "지점명", stations.loc[check.index, "name"])
check.round(3)

,지점명,기온 상관,"기온 차 (ERA5 - AWS 보간, °C)",강수 일치율,ERA5만 강수인 시간 비율
stn,,,,,
12103,군산휴(도),0.972,0.364,0.624,0.044
12209,서김제(도),0.973,0.710,0.594,0.045
12210,부안고려청자휴(도),0.965,0.957,0.576,0.064
12211,고인돌휴(도),0.967,0.312,0.539,0.061
12409,우곡3교(도),0.971,0.384,0.624,0.044
12410,부곡교(도),0.968,0.620,0.363,0.057
23202,여산휴(도),0.964,0.563,0.729,0.044
23203,삼례졸음(도),0.972,-0.017,0.633,0.039
23204,정읍휴(도),0.969,0.047,0.626,0.044


ERA5 2m 기온은 AWS 보간 기온과 상관이 0.95~0.98이고 평균 차이는 지점마다 -1.0 ~ +1.0°C입니다. AWS가 강수를 감지한 시간의 36~79%에서 ERA5에도 강수(0.1 mm/h 이상)가 있습니다. 시각 어긋남이나 단위 오류의 흔적은 없습니다.

### 6-3. 구간이 쓰는 격자

구간 233개의 중간점에서 가장 가까운 0.1° 격자를 지점과 같은 규칙으로 정합니다. 받은 격자가 없는 구간이 있으면 구간 자료를 만들지 않습니다. 가까운 다른 격자로 대신하면 일부 구간이 10~20km 떨어진 값을 쓰게 되고, 등급 차이가 격자 경계를 따라 생길 수 있기 때문입니다.

In [6]:
segments = gpd.read_file(config.PROCESSED_DIR / "segments.gpkg")
seg_cells = segments[["seg_id", "route", "lat", "lon"]].copy()
seg_cells["cell_lat"], seg_cells["cell_lon"] = zip(*[ex.cell_of(a, b) for a, b in zip(seg_cells["lat"], seg_cells["lon"])])
seg_cells["확보"] = [(a, b) in raw for a, b in zip(seg_cells["cell_lat"], seg_cells["cell_lon"])]
need = seg_cells.groupby(["cell_lat", "cell_lon"]).agg(
    구간수=("seg_id", "size"), 노선=("route", lambda r: ", ".join(sorted(set(r)))), 확보=("확보", "first")).reset_index()
print(f"구간이 쓰는 격자 {len(need)}개: 확보 {int(need['확보'].sum())}개, 미확보 {int((~need['확보']).sum())}개 "
      f"(구간 {int((~seg_cells['확보']).sum())}개)")
HOURS = TIMES[TIMES.minute == 0]  # 구간 변수는 정시만 있다 (02-2의 4장)
if seg_cells["확보"].all():
    aligned = {c: ex.align(raw[c], HOURS) for c in set(zip(seg_cells["cell_lat"], seg_cells["cell_lon"]))}
    seg_era5 = pd.concat([aligned[(r.cell_lat, r.cell_lon)].assign(seg_id=r.seg_id).rename_axis("time").reset_index()
                          for r in seg_cells.itertuples()], ignore_index=True)[["seg_id", "time", *ex.INST.values(), *ex.ACC.values()]]
    print("구간 ERA5 자료:", seg_era5.shape, "결측", int(seg_era5.isna().sum().sum()))
else:
    seg_era5 = None
    print("미확보 격자가 있어 구간 ERA5 자료는 만들지 않는다 (격자를 받아 data/external/era5land_cells/에 두고 다시 실행)")
need


구간이 쓰는 격자 25개: 확보 25개, 미확보 0개 (구간 0개)


구간 ERA5 자료: (1342080, 10) 결측 0


,cell_lat,cell_lon,구간수,노선,확보
0,35.3,126.6,5,서해안,True
1,35.4,126.6,9,서해안,True
2,35.4,126.7,5,서해안,True
3,35.5,126.7,13,서해안,True
4,35.5,126.8,11,호남선,True
5,35.6,126.7,13,서해안,True
6,35.6,126.8,7,호남선,True
7,35.6,126.9,9,호남선,True
8,35.7,126.7,5,서해안,True
9,35.7,126.8,9,서해안,True


### 6-4. 서해 해기차

찬 공기가 따뜻한 서해를 건너면 아래층이 불안정해져 눈구름이 발달합니다. 01-4의 6-3에서 내륙 노선의 Ice는 서·북서풍과 함께 나왔습니다. 이 조건의 세기를 해기차(해수면온도 - 850hPa 기온)로 나타냅니다.

| 항목 | 내용 |
|------|------|
| 자료 | NOAA OISST v2.1 (일별, 0.25°), ERA5 기압면 850hPa 기온·바람 (6시간) |
| 지점 | A(36.0N, 126.0E), B(36.5N, 125.5E). 서해 해상 두 점이고 모든 구간에 같은 시계열 |
| 계산 | dT = 전날 해수면온도 - 850hPa 기온. 해수면온도는 ERA5 격자 한 칸을 덮는 OISST 4개 격자의 평균 |
| 정렬 | 6시간 값 사이를 선형 보간 (02-1과 같은 규칙). 풍향은 0/360을 넘나들어 u·v 성분으로 보간 |
| 파일 | `data/external/haegicha_6h_kst.csv`. 저장소에는 없고 만드는 방법은 README 3장에 있음 |

기상청 예보기술 자료(손에 잡히는 예보기술 제50호, 3.3절)는 오대호의 850hPa 해기차 기준 13°C(Niziol 1987)를 우리나라에 그대로 쓸 수 없다고 보고, 2003~2017년 겨울 서해안 강설 사례 327건 중 315건(96.2%)이 15°C 이상이었다는 근거로 서해 임계값을 15°C로 제시합니다. 이 값은 칠발도 수온과 흑산도 850hPa 기온으로 잰 것이라 더 북쪽인 A·B 지점에 그대로 옮길 수는 없습니다. 쓰임새는 05-1의 2장에 있습니다.

In [7]:
sea = ex.load_haegicha(TIMES)
print(sea.shape, "결측", int(sea.isna().sum().sum()), "/ A·B dT 상관", round(sea["sea_dT_c_A"].corr(sea["sea_dT_c_B"]), 3))
sea.groupby(sea.index.to_period("M"))[["sea_sst_prev_c_A", "sea_t850_c_A", "sea_dT_c_A", "sea_dT_c_B"]].mean().round(1)


(34560, 10) 결측 0 / A·B dT 상관 0.992


,sea_sst_prev_c_A,sea_t850_c_A,sea_dT_c_A,sea_dT_c_B
2024-11,17.6,3.6,14.0,14.6
2024-12,12.6,-5.5,18.2,18.3
2025-01,8.8,-7.1,15.9,15.9
2025-02,6.0,-8.3,14.3,14.4
2025-11,15.5,2.6,12.9,13.3
2025-12,11.5,-3.5,15.1,15.4
2026-01,7.8,-7.5,15.3,15.7
2026-02,5.9,-2.5,8.5,9.1


## 저장

네 파일을 `data/processed/`에 저장합니다. `era5_10min.parquet`은 03-2·04-2·05-1에서, `era5_segment_cells.csv`와 `segment_era5_1h.parquet`은 05-2에서, `sea_10min.parquet`은 04-2·05-1에서 씁니다.

In [8]:
era5.to_parquet(config.PROCESSED_DIR / "era5_10min.parquet", index=False)
seg_cells.drop(columns=["lat", "lon"]).to_csv(config.PROCESSED_DIR / "era5_segment_cells.csv", index=False, encoding="utf-8-sig")
seg_path = config.PROCESSED_DIR / "segment_era5_1h.parquet"
if seg_era5 is not None:
    seg_era5.to_parquet(seg_path, index=False)
else:
    seg_path.unlink(missing_ok=True)  # 예전 실행의 구간 자료가 남아 05-2가 쓰지 않도록 지운다
sea.rename_axis("time").reset_index().to_parquet(config.PROCESSED_DIR / "sea_10min.parquet", index=False)
print("저장 완료")

저장 완료
